In [95]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random
from datasets import load_dataset
from transformers import AutoTokenizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Hardware Target:', device)


Hardware Target: cuda


In [96]:
print("Downloading Dataset...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
dataset = load_dataset('rotten_tomatoes')
def preprocess(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=64)
tokenized = dataset.map(preprocess, batched=True)
tokenized.set_format(type='torch', columns=['input_ids', 'label'])
train_loader = torch.utils.data.DataLoader(tokenized['train'], batch_size=256, shuffle=True)
test_loader = torch.utils.data.DataLoader(tokenized['test'], batch_size=256)
nlp_vocab_size = tokenizer.vocab_size


In [97]:
class ClassicalMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(nlp_vocab_size, 128)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 2)
    def forward(self, x):
        emb = self.embedding(x)
        pool = emb.mean(dim=1)
        return self.fc2(F.relu(self.fc1(pool)))
mlp = ClassicalMLP().to(device)
print(f"ClassicalMLP Parameters: {sum(p.numel() for p in mlp.parameters() if p.requires_grad):,}")
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)
print("\\n[Training MLP Sentiment Baseline...]")
mlp.train()
for epoch in range(8):
    total_loss = 0
    for batch in train_loader:
        x, y = batch['input_ids'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(mlp(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")
mlp.eval()
correct = sum((mlp(b['input_ids'].to(device)).argmax(1) == b['label'].to(device)).sum().item() for b in test_loader)
print(f"\\n--> Final MLP Accuracy on Test Set: {(correct/len(tokenized['test']))*100:.2f}%")


ClassicalMLP Parameters: 3,915,202
\n[Training MLP Sentiment Baseline...]
Epoch 1 Loss: 0.6942
Epoch 2 Loss: 0.6893
Epoch 3 Loss: 0.6837
Epoch 4 Loss: 0.6712
Epoch 5 Loss: 0.6486
Epoch 6 Loss: 0.6103
Epoch 7 Loss: 0.5647
Epoch 8 Loss: 0.5083
\n--> Final MLP Accuracy on Test Set: 67.54%


In [98]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:x.size(1), :].unsqueeze(0)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.d_k = d_model // heads
        self.heads = heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
    def forward(self, x, mask=None):
        B = x.size(0)
        q = self.w_q(x).view(B, -1, self.heads, self.d_k).transpose(1, 2)
        k = self.w_k(x).view(B, -1, self.heads, self.d_k).transpose(1, 2)
        v = self.w_v(x).view(B, -1, self.heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, -1, self.heads * self.d_k)
        return self.w_o(out)
class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads, d_ff=256, dropout=0.3):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.n1 = nn.LayerNorm(d_model)
        self.n2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.n1(x + self.drop(self.self_attn(x, mask)))
        x = self.n2(x + self.drop(self.ffn(x)))
        return x


In [99]:
class SentimentTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(nlp_vocab_size, 128)
        self.pos = PositionalEncoding(128)
        self.blocks = nn.ModuleList([TransformerBlock(128, heads=4) for _ in range(3)])
        self.fc = nn.Linear(128, 2)
    def forward(self, x, pad_mask):
        x = self.pos(self.embed(x))
        for block in self.blocks:
            x = block(x, pad_mask)
        pool = x.mean(dim=1)
        return self.fc(pool)


In [100]:
trans_model = SentimentTransformer().to(device)
print(f"SentimentTransformer Parameters: {sum(p.numel() for p in trans_model.parameters() if p.requires_grad):,}")
optimizer = torch.optim.Adam(trans_model.parameters(), lr=0.001)
print("\\n[Training Sentiment Transformer...]")
trans_model.train()
for epoch in range(8):
    total_loss = 0
    for batch in train_loader:
        x, y = batch['input_ids'].to(device), batch['label'].to(device)
        pad_mask = (x != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2).to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(trans_model(x, pad_mask), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")
trans_model.eval()
correct = 0
with torch.no_grad():
    for b in test_loader:
        x, y = b['input_ids'].to(device), b['label'].to(device)
        pad_mask = (x != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2).to(device)
        correct += (trans_model(x, pad_mask).argmax(1) == y).sum().item()
print(f"\\n--> Final Transformer Accuracy on Test Set: {(correct/len(tokenized['test']))*100:.2f}%")


SentimentTransformer Parameters: 4,304,514
\n[Training Sentiment Transformer...]
Epoch 1 Loss: 0.7455
Epoch 2 Loss: 0.6674
Epoch 3 Loss: 0.6003
Epoch 4 Loss: 0.5079
Epoch 5 Loss: 0.4368
Epoch 6 Loss: 0.3370
Epoch 7 Loss: 0.2603
Epoch 8 Loss: 0.1731
\n--> Final Transformer Accuracy on Test Set: 73.26%


In [101]:
vocab = list('0123456789+*=') + ['<pad>', '<bos>', '<eos>']
c2i = {c: i for i, c in enumerate(vocab)}; i2c = {i: c for c, i in c2i.items()}
pad_id, bos_id, eos_id = c2i['<pad>'], c2i['<bos>'], c2i['<eos>']
TRAIN_DIGITS = [1, 2, 3, 6]
TEST_DIGITS  = [4, 5]
class MathDataset(torch.utils.data.IterableDataset):
    def __init__(self, digit_lengths=None, max_len=32):
        self.digit_lengths = digit_lengths or TRAIN_DIGITS
        self.max_len = max_len
    def __iter__(self):
        while True:
            d_a, d_b = random.choice(self.digit_lengths), random.choice(self.digit_lengths)
            a, b = random.randint(0, 10**d_a - 1), random.randint(0, 10**d_b - 1)
            ans = a + b
            ans_str = str(ans)[::-1]
            prompt_str = f"{a}+{b}="
            seq = [bos_id] + [c2i[c] for c in prompt_str + ans_str] + [eos_id]
            ix = torch.full((self.max_len,), pad_id, dtype=torch.long)
            ly = torch.full((self.max_len,), pad_id, dtype=torch.long)
            ix[:len(seq)-1] = torch.tensor(seq[:-1])
            tar = seq[1:]
            for i in range(len(prompt_str)):
                tar[i] = pad_id
            ly[:len(seq)-1] = torch.tensor(tar)
            yield ix, ly
math_loader = torch.utils.data.DataLoader(MathDataset(), batch_size=256)


In [102]:
class MathGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(vocab), 256)
        self.pos = PositionalEncoding(256)
        self.blocks = nn.ModuleList([TransformerBlock(256, heads=8) for _ in range(4)])
        self.fc = nn.Linear(256, len(vocab))
    def forward(self, x, pad_mask=None):
        seq_len = x.size(1)
        causal = torch.tril(torch.ones(1, seq_len, seq_len)).bool().to(x.device)
        if pad_mask is not None: causal = causal & pad_mask
        x = self.pos(self.embed(x))
        for b in self.blocks: x = b(x, causal)
        return self.fc(x)


In [103]:
gpt = MathGPT().to(device)
print(f"MathGPT Parameters: {sum(p.numel() for p in gpt.parameters() if p.requires_grad):,}")
opt = torch.optim.Adam(gpt.parameters(), lr=0.001)
max_steps = 3000
print(f"\\n[Initiating GPT Math Training for {max_steps} steps...]")
gpt.train()
t_loss = 0
for step, (x, y) in enumerate(math_loader):
    if step >= max_steps: break
    x, y = x.to(device), y.to(device)
    pm = (x != pad_id).unsqueeze(1).unsqueeze(2)
    opt.zero_grad()
    loss = F.cross_entropy(gpt(x, pm).reshape(-1, len(vocab)), y.reshape(-1), ignore_index=pad_id)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
    opt.step()
    t_loss += loss.item()
    if (step + 1) % 500 == 0:
        print(f"[{step+1}/{max_steps}] Loss: {t_loss/500:.4f}")
        t_loss = 0


MathGPT Parameters: 1,591,312
\n[Initiating GPT Math Training for 3000 steps...]
[500/3000] Loss: 1.2067
[1000/3000] Loss: 0.5441
[1500/3000] Loss: 0.2244
[2000/3000] Loss: 0.1908
[2500/3000] Loss: 0.1460
[3000/3000] Loss: 0.1365


In [104]:
gpt.eval()
def test_manual(prompt):
    print(f"\\n[Testing Prompt]: {prompt}")
    x = torch.tensor([[bos_id] + [c2i[c] for c in prompt]], device=device)
    with torch.no_grad():
        for _ in range(30):
            next_tok = gpt(x)[:, -1, :].argmax(dim=-1, keepdim=True)
            if next_tok.item() in [eos_id, pad_id]: break
            x = torch.cat([x, next_tok], dim=1)
    out = "".join([i2c[t.item()] for t in x[0] if t.item() not in [bos_id, eos_id, pad_id]])
    pred_ans = out.replace(prompt, "")
    print(f"  Raw Decoder Output  : {pred_ans}")
    print(f"  Final Human Readable: {prompt}{pred_ans[::-1]}")
test_manual("33+44=")
test_manual("1234+5678=")
test_manual("12345+54321=")
test_manual("99999+1=")
test_manual("555555+444444=")


\n[Testing Prompt]: 33+44=
  Raw Decoder Output  : 77
  Final Human Readable: 33+44=77
\n[Testing Prompt]: 1234+5678=
  Raw Decoder Output  : 27061
  Final Human Readable: 1234+5678=16072
\n[Testing Prompt]: 12345+54321=
  Raw Decoder Output  : 658255
  Final Human Readable: 12345+54321=552856
\n[Testing Prompt]: 99999+1=
  Raw Decoder Output  : 000001
  Final Human Readable: 99999+1=100000
\n[Testing Prompt]: 555555+444444=
  Raw Decoder Output  : 999999
  Final Human Readable: 555555+444444=999999


In [105]:
def gen_prob(digits):
    a, b = random.randint(10**(digits-1), 10**digits - 1), random.randint(10**(digits-1), 10**digits - 1)
    op = '+'
    ans = a + b
    return f"{a}{op}{b}=", str(ans)[::-1]
print("\\n[Evaluating Algorithmic Interpolation Intelligence]")
print("Training digits: {1,2,3,6} -- Testing on HELD-OUT 4 and 5-digit problems!\n")
for num_digits in [2, 3, 4, 5, 6]:
    correct = 0
    total = 50
    label = "HELD OUT" if num_digits in TEST_DIGITS else "trained"
    print(f"\\n--- {num_digits}-Digit Testing Phase ({label}) ---")
    for _ in range(total):
        prompt, expected = gen_prob(num_digits)
        x = torch.tensor([[bos_id] + [c2i[c] for c in prompt]], device=device)
        with torch.no_grad():
            for _ in range(30):
                next_tok = gpt(x)[:, -1, :].argmax(dim=-1, keepdim=True)
                if next_tok.item() in [eos_id, pad_id]: break
                x = torch.cat([x, next_tok], dim=1)
        out = "".join([i2c[t.item()] for t in x[0] if t.item() not in [bos_id, eos_id, pad_id]])
        pred_ans = out.replace(prompt, "")
        if pred_ans == expected:
            correct += 1
    print(f"Accuracy exactly matched: {correct}/{total} ({(correct/total)*100:.2f}%)")


\n[Evaluating Algorithmic Interpolation Intelligence]
Training digits: {1,2,3,6} -- Testing on HELD-OUT 4 and 5-digit problems!

\n--- 2-Digit Testing Phase (trained) ---
Accuracy exactly matched: 50/50 (100.00%)
\n--- 3-Digit Testing Phase (trained) ---
Accuracy exactly matched: 49/50 (98.00%)
\n--- 4-Digit Testing Phase (HELD OUT) ---
Accuracy exactly matched: 0/50 (0.00%)
\n--- 5-Digit Testing Phase (HELD OUT) ---
Accuracy exactly matched: 0/50 (0.00%)
\n--- 6-Digit Testing Phase (trained) ---
Accuracy exactly matched: 0/50 (0.00%)
